# ⚡ Duino API — Google Colab
**Hyperscale AI inference — Gemma 4 · Multi-GPU · Streaming · React UI**

> 💡 `Runtime → Change runtime type → T4 GPU`  
> 🔑 Add `HF_TOKEN` to Colab Secrets (🔑 sidebar)  
> 🌐 Public HTTPS links auto-generated after `start()`

---

## 📦 Cell 1 — Clone & Install

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/jalalmansour/Duino_API.git'
REPO_DIR = '/content/Duino_API'

if not os.path.exists(REPO_DIR):
    print('Cloning Duino API...')
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Pulling latest...')
    !git -C {REPO_DIR} pull --quiet

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

!pip install --upgrade setuptools wheel pip -q
!pip install -r /content/Duino_API/requirements.txt -q

r = subprocess.run(['pip','install','-e',REPO_DIR,'-q','--no-build-isolation'],
                   capture_output=True, text=True)
if r.returncode != 0:
    subprocess.run(['pip','install',REPO_DIR,'-q'])

print('\n✅ Setup complete!')

## 🔑 Cell 2 — HuggingFace Token

In [ ]:
import os
# Add HF_TOKEN via 🔑 Colab Secrets sidebar (recommended)
# os.environ['HF_TOKEN'] = 'hf_...'
print(f"HF_TOKEN        : {'✅ set' if os.environ.get('HF_TOKEN') else '⚠️  NOT SET'}")
print(f"NGROK_AUTHTOKEN : {'✅ set' if os.environ.get('NGROK_AUTHTOKEN') else '⚠️  not set (Colab proxy used)'}")

## 🚀 Cell 3 — Start Platform

In [ ]:
import sys, os, time
REPO_DIR = '/content/Duino_API'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from studio.backend.colab import start

result  = start(model='gemma-4-2b', api_port=8000, ui_port=3000, expose=True)
api_url = result['api_url']
ui_url  = result['ui_url']
api_key = result['api_key']
embed_html = result['embed_html']

print('\n' + '═'*62)
print('  🌐 PUBLIC LINKS — share with anyone')
print('═'*62)
print(f'  📡 API      : {api_url}')
print(f'  📖 Docs     : {api_url}/docs')
print(f'  🎨 UI       : {ui_url}')
print(f'  🔑 API Key  : {api_key}')
print('═'*62)

## 🎨 Cell 4 — Embed UI & API Docs

In [ ]:
from google.colab import output as colab_output
print('🎨 React Chat UI (port 3000)')
colab_output.serve_kernel_port_as_iframe(3000, height=700, width='100%')

In [ ]:
from google.colab import output as colab_output
print('📖 API Docs / Swagger (port 8000)')
colab_output.serve_kernel_port_as_iframe(8000, height=900, width='100%')

## 🧪 Cell 5 — Test API

In [ ]:
import requests, json

resp = requests.post(f'{api_url}/v1/chat/completions',
    headers={'X-API-Key': api_key},
    json={'model':'gemma-4-2b',
          'messages':[{'role':'user','content':'What is the capital of France?'}],
          'max_tokens':64})
print('Answer:', resp.json()['choices'][0]['message']['content'])

print('\nStreaming:')
with requests.post(f'{api_url}/v1/chat/completions',
    headers={'X-API-Key': api_key},
    json={'model':'gemma-4-2b',
          'messages':[{'role':'user','content':'Write a haiku about the ocean.'}],
          'max_tokens':64,'stream':True}, stream=True) as r:
    for line in r.iter_lines():
        if line:
            l = line.decode()
            if l.startswith('data: ') and l[6:] != '[DONE]':
                print(json.loads(l[6:])['choices'][0].get('delta',{}).get('content',''),
                      end='', flush=True)
print()

## 💬 Cell 6 — Multi-turn Chat

In [ ]:
import requests
sess = requests.post(f'{api_url}/v1/sessions',
    headers={'X-API-Key': api_key},
    json={'model_id': 'gemma-4-2b'}).json()
session_id = sess['session_id']

def chat(msg):
    return requests.post(f'{api_url}/v1/chat/completions',
        headers={'X-API-Key': api_key},
        json={'model':'gemma-4-2b',
              'messages':[{'role':'user','content':msg}],
              'session_id':session_id,'max_tokens':256}
    ).json()['choices'][0]['message']['content']

print('AI:', chat('My name is Jalal. Remember it.'))
print('AI:', chat('What is my name?'))

## ⚛️ Cell 7 — Vite / Next.js Inside Colab

In [ ]:
import subprocess, threading, time, os
from google.colab import output as colab_output

REACT_PORT = 4000
REACT_DIR  = '/content/my-react-app'
if not os.path.exists(REACT_DIR):
    !npm create vite@latest {REACT_DIR} -- --template react -y 2>/dev/null
    !npm install --prefix {REACT_DIR} --silent

def _vite():
    subprocess.run(['npm','run','dev','--','--host','0.0.0.0','--port',str(REACT_PORT)],
                   cwd=REACT_DIR)
threading.Thread(target=_vite, daemon=True).start()
time.sleep(6)
colab_output.serve_kernel_port_as_iframe(REACT_PORT, height=600, width='100%')

## ♾️ Cell 9 — Live Monitor + Keep Alive (NEVER STOPS)
_Real-time GPU stats, API health, multi-GPU support. Only your STOP (■) button terminates this._

In [ ]:
import sys, os, time, threading, requests
from IPython.display import display, Javascript
from google.colab import output as colab_output

REPO_DIR = '/content/Duino_API'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

# ── 1. Print public links ─────────────────────────────────────────────────────
print('═'*62)
print('  🌐 PUBLIC LINKS')
print('═'*62)
print(f'  📡 API  : {api_url}')
print(f'  📖 Docs : {api_url}/docs')
print(f'  🎨 UI   : {ui_url}')
print(f'  🔑 Key  : {api_key}')
print('═'*62)

# ── 2. Embed React UI ─────────────────────────────────────────────────────────
colab_output.serve_kernel_port_as_iframe(3000, height=1200, width='100%')

# ── 3. JavaScript anti-disconnect (simulates activity every 60s) ──────────────
display(Javascript("""
function duinoKeepAlive() {
  try {
    // Reset Colab idle timer by dispatching mouse activity
    document.dispatchEvent(new MouseEvent('mousemove', {bubbles:true}));
    // Click Connect button if session dropped
    var btn = document.querySelector('#top-toolbar > colab-connect-button');
    if (btn) btn.click();
  } catch(e) {}
  setTimeout(duinoKeepAlive, 60000); // every 60 seconds
}
duinoKeepAlive();
console.log('[Duino] Anti-disconnect watchdog active');
"""))

# ── 4. Live CLI monitor (in-place updating like htop) ─────────────────────────
from studio.monitor import Monitor

monitor = Monitor(
    api_url  = api_url,
    api_key  = api_key,
    interval = 5,    # refresh every 5 seconds
    ping_api = True,
)

print()
print('[Monitor] Starting live dashboard...')
print('[Monitor] This cell runs FOREVER — press STOP (■) to terminate.')
print()

# run() blocks forever — catches ALL exceptions internally
monitor.run()